In [5]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import traceback
from matplotlib.animation import FFMpegWriter

BASE_DIR = './Data'

GIF_DIR = os.path.join(BASE_DIR, 'gifs_no_peaks')

# Sliding Window Parameters
WINDOW_SIZE = 30000  # Number of data points in the window (e.g., 1 second if data is at 10kHz)
STEP_SIZE = 5000     # Number of data points to slide the window by (controls overlap and speed)

DATA_COLUMN = '0'   
TIME_COLUMN = 'Time'

# Animation Parameters
FPS = 1             # Frames per second for the output GIF
INTERVAL = 500       # Delay between frames in milliseconds (1000 / FPS)

os.makedirs(GIF_DIR, exist_ok=True)
print(f"Base directory: {os.path.abspath(BASE_DIR)}")
print(f"GIF output directory: {os.path.abspath(GIF_DIR)}")



Base directory: c:\Users\Andrew\Documents\Acoustic-Space-Boiling\Data
GIF output directory: c:\Users\Andrew\Documents\Acoustic-Space-Boiling\Data\gifs_no_peaks


In [6]:
def create_gif_sliding_window(csv_file, output_gif):
    """ Creates an animated GIF (sliding window only) for a single CSV file. """
    fig = None 
    try:
        # --- Load Data ---
        try:
            data = pd.read_csv(csv_file)
            if TIME_COLUMN not in data.columns:
                print(f"      Skipping {os.path.basename(csv_file)}: Missing '{TIME_COLUMN}' column.")
                return
            if DATA_COLUMN not in data.columns:
                print(f"      Skipping {os.path.basename(csv_file)}: Missing '{DATA_COLUMN}' column.")
                return

            data = data.set_index(TIME_COLUMN)

        except pd.errors.EmptyDataError:
            print(f"      Skipping {os.path.basename(csv_file)}: File is empty.")
            return
        except Exception as e:
            print(f"      Failed to read CSV {os.path.basename(csv_file)}: {e}")
            # traceback.print_exc() # Uncomment for detailed read errors
            return # Skip this file

        time = data.index.to_numpy()
        signal_data = data[DATA_COLUMN].to_numpy()

        # --- Check Data Length ---
        if len(time) <= WINDOW_SIZE:
            print(f"      Skipping {os.path.basename(csv_file)}: Not enough data points ({len(time)}) for window size ({WINDOW_SIZE}).")
            return
        if len(time) != len(signal_data):
             print(f"      Skipping {os.path.basename(csv_file)}: Mismatch between time ({len(time)}) and data ({len(signal_data)}) points.")
             return

        # --- Setup Plot ---
        fig, ax = plt.subplots(figsize=(12, 7)) 

        # Only plot the main data line
        line, = ax.plot([], [], lw=1.5, color='dodgerblue', label=f'Accelerometer Value')

        # Set initial dynamic Y limits with padding
        y_min, y_max = signal_data.min(), signal_data.max()
        y_range = y_max - y_min
        y_pad = 0.1 * y_range if y_range > 1e-9 else 0.1
        ax.set_ylim(y_min - y_pad, y_max + y_pad)

        # Setup plot labels and appearance
        ax.set_title(f"Sliding Window Visualization\n{os.path.basename(csv_file)}", fontsize=14)
        ax.set_xlabel("Time (s)", fontsize=12)
        ax.set_ylabel(f"Accelerometer Value", fontsize=12)
        ax.grid(True, linestyle='--', alpha=0.6)
        ax.legend()
        fig.tight_layout()

        # --- Animation Update Function ---
        def update(frame_index):
            """ Updates the plot for each frame of the animation. """
            start_index = frame_index
            end_index = start_index + WINDOW_SIZE

            # Ensure end_index doesn't exceed array bounds
            if end_index > len(time):
                 end_index = len(time)

            if start_index >= end_index:
                return line, 

            # Get data for the current window
            time_window = time[start_index:end_index]
            signal_data_window = signal_data[start_index:end_index]

            # Update the main line plot data
            line.set_data(time_window, signal_data_window)

            # Update x-axis limits to follow the window
            if len(time_window) > 0:
                 ax.set_xlim(time_window[0], time_window[-1])

            # Return the artists that were updated 
            return line,

        # --- Create and Save Animation ---
        # Calculate frames based on the starting index of each window
        frames = range(0, len(time) - WINDOW_SIZE + 1, STEP_SIZE)

        # Check if the frames iterator will be empty (e.g., if step_size is too large)
        if not list(frames):
             print(f"      Skipping {os.path.basename(csv_file)}: No valid frames generated with window/step size.")
             plt.close(fig) # Close the figure explicitly
             return

        # Re-create the frames range as it was consumed by list()
        frames = range(0, len(time) - WINDOW_SIZE + 1, STEP_SIZE)

        # Use blit=True for potentially faster rendering, requires update func to return iterable of artists
        ani = animation.FuncAnimation(fig, update, frames=frames, blit=True, interval=INTERVAL)

        # Save the GIF using Pillow writer
        print(f"  Saving GIF to: {output_gif}")
        writer = FFMpegWriter(fps=FPS)
        ani.save(output_gif, writer=writer)

    except Exception as e:
        # Catch any unexpected errors during plotting or animation
        print(f"    !! Unexpected Error processing {os.path.basename(csv_file)}: {e}")
        traceback.print_exc() # Print detailed traceback for debugging

    finally:
        if fig:
            plt.close(fig)


In [7]:
def process_all_files_sliding_window():
    """ Walks through BASE_DIR and creates sliding window GIFs for each CSV file. """
    print(f"Starting processing in: {BASE_DIR}")
    print(f"Outputting GIFs to: {GIF_DIR}")

    file_count = 0
    processed_count = 0
    error_count = 0

    for root, dirs, files in os.walk(BASE_DIR):
        # IMPORTANT: Skip the GIF output directory itself to prevent recursion
        if os.path.abspath(root).startswith(os.path.abspath(GIF_DIR)):
            print(f"  Skipping GIF output directory: {root}")
            continue

        # Also skip any directory named 'gifs' that isn't the target GIF_DIR
        if 'gifs' in dirs and os.path.abspath(os.path.join(root, 'gifs')) != os.path.abspath(GIF_DIR):
             print(f"  Skipping unrelated 'gifs' directory: {os.path.join(root, 'gifs')}")
             dirs[:] = [d for d in dirs if d != 'gifs']


        for file in files:
            if file.lower().endswith('.csv'):
                file_count += 1
                full_path = os.path.join(root, file)

                # Create a relative path from BASE_DIR to maintain structure in GIF_DIR
                try:
                    rel_path_from_base = os.path.relpath(full_path, BASE_DIR)
                except ValueError:
                     print(f"      Skipping {full_path}: Cannot determine relative path from {BASE_DIR}")
                     error_count += 1
                     continue

                gif_filename = os.path.splitext(rel_path_from_base)[0] + '.mp4'
                gif_path = os.path.join(GIF_DIR, gif_filename)

                # Create necessary subdirectories within GIF_DIR
                os.makedirs(os.path.dirname(gif_path), exist_ok=True)

                print(f"\nProcessing [{file_count}]: {rel_path_from_base}")
                try:
                    create_gif_sliding_window(full_path, gif_path)
                    processed_count += 1
                except Exception as e:
                    # Catch any unexpected errors from create_gif itself
                    print(f"  !! Top-Level Error creating GIF for {rel_path_from_base}: {e}")
                    traceback.print_exc()
                    error_count += 1

    # --- Summary ---
    print(f"\n--- Processing Complete ---")
    print(f"Total CSV files found: {file_count}")
    print(f"Successfully processed: {processed_count}")
    # Calculate skipped/failed count accurately
    skipped_or_failed = file_count - processed_count
    print(f"Files skipped or failed: {skipped_or_failed}")


In [8]:
process_all_files_sliding_window()

Starting processing in: ./Data
Outputting GIFs to: ./Data\gifs_no_peaks

Processing [1]: CombinedClusteringWaveletData.csv
      Skipping CombinedClusteringWaveletData.csv: Missing 'Time' column.

Processing [2]: After_May\MATLAB 1-00 PM Fri, Jun 28, 2024 Run8 .csv
  Saving GIF to: ./Data\gifs_no_peaks\After_May\MATLAB 1-00 PM Fri, Jun 28, 2024 Run8 .mp4

Processing [3]: After_May\MATLAB 1-02 PM Thu, Nov 7, 2024 Run8 .csv
  Saving GIF to: ./Data\gifs_no_peaks\After_May\MATLAB 1-02 PM Thu, Nov 7, 2024 Run8 .mp4

Processing [4]: After_May\MATLAB 1-04 PM Thu, Aug 22, 2024 Run2.csv
  Saving GIF to: ./Data\gifs_no_peaks\After_May\MATLAB 1-04 PM Thu, Aug 22, 2024 Run2.mp4

Processing [5]: After_May\MATLAB 1-04 PM Tue, Sep 10, 2024 Run9 .csv
  Saving GIF to: ./Data\gifs_no_peaks\After_May\MATLAB 1-04 PM Tue, Sep 10, 2024 Run9 .mp4

Processing [6]: After_May\MATLAB 1-05 PM Thu, Aug 22, 2024 Run3 .csv
  Saving GIF to: ./Data\gifs_no_peaks\After_May\MATLAB 1-05 PM Thu, Aug 22, 2024 Run3 .mp4

Pr